# LVP Colab Demo

Edge-preprocess a video into a compact `.lvp` package, then (optionally) query an LLM.

**Runtime:** enable a CPU runtime. Install FFmpeg in the first code cell.

In [ ]:
# !apt-get -qq install -y ffmpeg
# !pip install -q git+https://github.com/Girish011/lvp-package.git
import subprocess, shutil
print('ffmpeg:', shutil.which('ffmpeg'))
subprocess.run(['ffmpeg', '-version'], check=False)

In [ ]:
import subprocess
from pathlib import Path

sample = Path('demo.mp4')
if not sample.exists():
    subprocess.run([
        'ffmpeg', '-f', 'lavfi', '-i', 'smptebars=size=640x360:rate=25',
        '-f', 'lavfi', '-i', 'sine=frequency=440:duration=4',
        '-t', '4', '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-shortest', '-y', str(sample)
    ], check=True)
print('bytes', sample.stat().st_size)

In [ ]:
import lvp
pkg = lvp.process('demo.mp4', output='demo.lvp', profile='balanced', transcribe=False)
print(pkg.summary())
print('Saved demo.lvp — upload this instead of the raw MP4 when calling vision APIs.')

## Optional: query OpenAI

Set `OPENAI_API_KEY` in Colab secrets / environment, then run:

In [ ]:
import os
if os.environ.get('OPENAI_API_KEY'):
    from lvp.providers import OpenAIProvider
    print(OpenAIProvider().query(pkg, 'Describe this video in one sentence.'))
else:
    print('Set OPENAI_API_KEY to query a model.')